# Kraken Futures API Interaction Example

## Setup

In [ ]:
import os

from dotenv import load_dotenv

from kraken_kit import FuturesConnector

load_dotenv()

futures = FuturesConnector(
    api_key=os.environ["KRAKEN_FUTURES_API_KEY"],
    api_secret=os.environ["KRAKEN_FUTURES_API_SECRET"],
)

## Account Balance

In [ ]:
futures.get_accounts()

In [ ]:
futures.get_balance("USDC")

In [ ]:
futures.get_balance("USDC")['collateral']

## Tickers and Instruments

In [ ]:
instruments = futures.get_instruments()
print(f"{len(instruments)} instruments available")
for inst in instruments[:10]:
    print(f"  {inst['symbol']} - {inst.get('type', '')}")

In [ ]:
symbol = "PF_XBTUSD"
ticker = futures.get_ticker(symbol)
ticker

In [ ]:
mid_price = (ticker["bid"] + ticker["ask"]) / 2

## Margin Mode and Leverage

In [ ]:
futures.get_leverage_preferences()

In [ ]:
leverage = 2
futures.set_leverage(symbol, leverage)

In [ ]:
futures.get_leverage_preferences()

In [ ]:
futures.set_cross_margin(symbol)

In [ ]:
futures.get_leverage_preferences()

## Place a Market Order

In [ ]:
side = "buy"
qty = 10 / mid_price

futures.place_order(symbol, side, qty)

In [ ]:
side = "buy"
qty = 10 / mid_price

futures.place_order(symbol, side, qty)

## Open Positions

In [ ]:
positions = futures.get_open_positions()
positions

In [ ]:
position = futures.get_open_position(symbol)
position

## Stop-Loss + Take-Profit Together

Both are reduce-only and they reference an existing position, no fund locking conflict.

In [ ]:
position_size = position["size"]

side = "sell"
stop_loss_price = 60000
take_profit_price = 80000

futures.place_stop_loss(symbol, side, position_size, stop_loss_price=stop_loss_price)
futures.place_take_profit(symbol, side, position_size, take_profit_price=take_profit_price)

## Trailing Stop

Unlike stop-loss and take-profit, trailing stop is NOT forced reduce-only - you must set it explicitly.
The stop price trails the market by a deviation you define (percentage or absolute amount).

In [ ]:
side = "sell"
qty = position["size"]
max_deviation = 5                 # trail by 5%
deviation_unit = "PERCENT"        # or "QUOTE_CURRENCY" for absolute amount

futures.place_trailing_stop(
    symbol, side, qty,
    max_deviation=max_deviation,
    deviation_unit=deviation_unit,
    reduce_only=True,                 # not forced - you choose
)

## Place a Limit Order

In [ ]:
side = "sell"
qty = 10.0 / mid_price
price = 80000

limit_id = futures.place_order(symbol, side, qty, price=price)
print(f"Limit order: {limit_id}")

# Default is lmt (limit, GTC). Other order types for execution control:
# futures.place_order(symbol, side, qty, price=price, order_type="ioc")   # Immediate-or-cancel
# futures.place_order(symbol, side, qty, price=price, order_type="fok")   # Fill-or-kill
# futures.place_order(symbol, side, qty, price=price, order_type="post")  # Post-only

## Entry order + SL + TP + TSL all in one

In [ ]:
side = "buy"
qty = 10 / mid_price

futures.place_bracket_order(
    symbol, 
    "buy", 
    qty,
    # price=70000,
    stop_loss_price=65000,
    take_profit_price=90000,
    # trailing_stop_deviation=500,
    # trailing_stop_deviation_unit="QUOTE_CURRENCY", # 500 USD trail
)

## Order batch


In [ ]:
side = "buy"
qty = futures.format_qty(symbol, 10 / mid_price)
entry_price = 75000
sl_price = 72000
tp_price = 100000

result = futures.place_order_batch([
    # Entry: limit buy
    {"order": "send", "order_tag": "entry", "orderType": "lmt", "symbol": symbol,
     "side": side, "size": qty, "limitPrice": entry_price},
    # Stop-loss: reduce-only
    {"order": "send", "order_tag": "sl", "orderType": "stp", "symbol": symbol,
     "side": "sell", "size": qty,
     "stopPrice": sl_price, "limitPrice": sl_price * 0.99,
     "reduceOnly": True},
    # Take-profit: reduce-only
    {"order": "send", "order_tag": "tp", "orderType": "take_profit", "symbol": symbol,
     "side": "sell", "size": qty,
     "stopPrice": tp_price, "limitPrice": tp_price,
     "reduceOnly": True},
])
result["batchStatus"]

## Edit Order

In [ ]:
oid = limit_id

edit_result = futures.edit_order(oid, symbol=symbol, limit_price=90000)
print(edit_result)

## Cancel Orders

In [ ]:
order_id = edit_result['editStatus']['orderId']
futures.cancel_order(order_id)

In [ ]:
futures.cancel_all()

## Dead Man's Switch

Cancels all orders if no heartbeat is received within the timeout.
For exmaple call every 60s with a 120s timeout.

In [ ]:
# Activate with 120s timeout
futures.cancel_all_after(120)

# Deactivate
futures.cancel_all_after(0)